In [32]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


In [33]:
# Reading the saved data pickle file
df  = pd.DataFrame()
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/sentiment.pkl', compression=None)

In [ ]:
df["articles"].describe()
df = df.join(df_sentiment)
df["vix"] = df["adj close"].apply(np.int)
df["vix_ln_return"]=np.log(df["VIX"]/df["VIX"].shift(1))
df['sentiment'] = df[["sentiment"]].apply(pd.to_numeric)
df["sentiment_ewm"]=df["sentiment"].ewm(10).mean()
df['date'] = pd.date_range(start='25/9/2019', periods=len(df), freq='D')

In [ ]:
df["close", "vix_ln_return", ""]

In [ ]:
sentence = 'The shooting in Paris killed two incon'
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
sid
ss = sid.polarity_scores(sentence)
ss

In [ ]:
df.loc['2020-06-01':'2021-07-16']

In [ ]:
#df.diff().hist(color="k", alpha=0.5, bins=50);
names = ['negative','positive']
values = [np.sum(df['neg'].apply(pd.to_numeric)), np.sum(df['pos'].apply(pd.to_numeric))]

plt.figure(figsize=(9, 3))

#plt.subplot(131)
plt.bar(names, values)

In [ ]:
values = [np.sum(df['neg']), np.sum(df['neu']), np.sum(df['pos'])]

plt.figure(figsize=(5, 5))
plt.pie(values,labels=["neg","neu","pos"] ,autopct='%1.1f%%')
plt.legend()

values = [np.sum(df['neg']), np.sum(df['pos'])]
plt.figure(figsize=(5, 5))
plt.pie(values,labels=["neg","pos"] ,autopct='%1.1f%%')
plt.legend()
# # df[["neg","neu","pos"]].plot(y='neg', figsize=(5, 5), kind="pie")

In [ ]:
df.plot(x="vix_ln_return", y="sentiment", kind="scatter")

In [ ]:
df

In [ ]:
# function to add value labels
# function to add value labels
def addlabels(x,y):
    for i in range(len(x)):
        plt.text(i,1,y[i])

names = ['negative','neutral', 'positive']
neg=df['neg']
values = [np.sum(neg), np.sum(df['neu']), np.sum(df['pos'])]
#addlabels(names, values)
#plt.text(1, values[1], values[1], ha = 'center')
plt.figure()#figsize=(8, 3),
plt.xlabel("Sentiment Type")
plt.ylabel("Percentage of words")
plt.bar(names, values)

In [ ]:
df[['sentiment']] = df[["sentiment"]].apply(pd.to_numeric)

In [ ]:
df[["sentiment"]]

In [ ]:
# Creating plot
df[['sentiment', 'sentiment_ewm']].boxplot()
# plt.boxplot(data)
# names=['sentiment']
# plt.xticks([1], ['sentiment'])
# plt.bar(names, data)
# # # show plot
# plt.show()

In [ ]:
# Creating plot
data=df[['vix_ln_return']]
plt.boxplot(data)

plt.xticks([1], ['VIX'])
#plt.bar(names, values)
# # show plot
plt.show()

In [ ]:
df["sentiment"].dropna(how='all', inplace = True)#values#.to_string().(" ")
#df[['date' 'sentiment']] = pd.DataFrame([ x.split(' ') for x in df['_id'].tolist() ])


In [ ]:
df[['sentiment']]

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-07-15']

In [ ]:
df_part3["log_VIX"]

In [ ]:

# dfplot = df_part1[['VIX']].plot(figsize=(10, 5),title='2019-09-25 to 2020-02-14')
# fig = dfplot.get_figure()

# dfplot = df_part2[['VIX']].plot(figsize=(10, 5),title='2020-02-15 to 2020-05-31')
# fig = dfplot.get_figure()
dfplot = df_part3['log_VIX'].plot(figsize=(10, 5),title='2020-06-01 to 2021-07-15')
fig = dfplot.get_figure()

# dfplot = df_part3['VIX'].plot(figsize=(10, 5),title='2020-06-01 to 2021-07-15')
# fig = dfplot.get_figure()



df_part3[['sentiment']] =df_part3[['sentiment']].apply(pd.to_numeric)
df_part3[['sentiment_ewm']].plot(figsize=(10, 5),title='2020-06-01 to 2021-07-15')
dfplot = df[['sentiment']].plot(figsize=(10, 5),title='2020-06-01 to 2021-07-15')
fig = dfplot.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentiment.png")
df_part3[['sentiment']].ewm(span=20).mean().plot()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentiment_ewm.png")



In [ ]:
df_part3["sentiment"]

In [ ]:
import matplotlib.pyplot as plt
import datetime as dt
import matplotlib.dates as mdates

#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(22,10))
l1 = ax1.plot(df['VIX'], color='#ff7f50')
ax1.set_xlabel("Date",fontsize=20)
ax1.xaxis_date()
ax1.set_title("Sentiment and VIX", size=26)
ax1.set_ylabel("VIX", size=20)
# index=['sentiment', 'VIX']


# starting_date = dt.datetime(2019, 11, 15)
# starting_date1 = mdates.date2num(starting_date)
# pointing_arrow =dt.datetime(2020, 2, 15)
# pointing_arrow1 = mdates.date2num(pointing_arrow)
# ax1.annotate('Part \n One', xy=(pointing_arrow1, 15), xytext=(starting_date1,30),
#             arrowprops=dict(facecolor='#e50000', shrink=0.1),
#             )

# starting_date2 = dt.datetime(2020, 8, 15)
# starting_date3 = mdates.date2num(starting_date2)
# pointing_arrow3 =dt.datetime(2020, 6, 1)
# pointing_arrow4 = mdates.date2num(pointing_arrow3)
# ax1.annotate('Part \n Two', xy=(pointing_arrow4, 25), xytext=(starting_date3,15),
#             arrowprops=dict(facecolor='#e50000', shrink=0.1),
#             )



#ax1.axhline(y=15, color='b', linestyle='--')
# starting_date = dt.datetime(2020, 2, 15)
# starting_date1 = mdates.date2num(starting_date)
# ax1.axvline(starting_date1, 0, 1, color='b', linestyle='--')
# starting_date2 = dt.datetime(2020, 6, 1)
# starting_date3 = mdates.date2num(starting_date2)
# ax1.axvline(starting_date3, 0, 1, color='b', linestyle='--')

ax2 = ax1.twinx()
ax2.xaxis_date()
l2 = ax2.plot(df['sentiment'].ewm(span=20).mean())#, color='#f97306')# color='#e50000')
ax2.set_ylim([50, 100])
ax2.set_ylabel("sentiment", size=20)

labels=['VIX', 'sentiment']
fig.legend([l1, l2], labels=labels, loc='right', bbox_to_anchor=(0.9, 0.85))

fig.autofmt_xdate()
fig = ax2.get_figure()

#fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentimentVIX.png")

In [ ]:
import matplotlib.pyplot as plt
import datetime as dt
import matplotlib.dates as mdates

fig, ax1 = plt.subplots(figsize=(20,10))
l1 = ax1.plot(df_part3['VIX'], color='#ff7f50')
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("Sentiment and VIX \n 2020-06-01 to 2021-07-16", size=26)
ax1.set_ylabel("VIX", size=20)

ax2 = ax1.twinx()
l2 = ax2.plot(df_part3['sentiment'].ewm(span=20).mean())#, color='#f97306')# color='#e50000')
ax2.set_ylim([50, 100])
ax2.set_ylabel("sentiment", size=20)

labels=['VIX', 'sentiment']
fig.legend([l1, l2], labels=labels, loc='right', bbox_to_anchor=(0.9, 0.85))

fig.autofmt_xdate()
fig = ax2.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentimentVIXpart3.png")

In [ ]:

# dfplot = df_part1[['VIX']].plot(figsize=(10, 5),title='2019-09-14 to 2020-02-14')
# fig = dfplot.get_figure()   

# dfplot = df_part2[['VIX']].plot(figsize=(10, 5),title='2019-09-25 to 2020-02-14')
# fig = dfplot.get_figure()   

# dfplot = df_part3[['VIX']].plot(figsize=(10, 5),title='2019-09-25 to 2020-02-14')
# fig = dfplot.get_figure()   

fig, axs = plt.subplots(2,figsize=(20,15))
fig.suptitle('VIX and Sentiment', size=26)
fig.autofmt_xdate()
# axs[0].set_ylabel("sentiment", size=20)
# axs[0].plot(df_part1[['sentiment']])
# axs[0].plot(df_part2[['sentiment']])
# axs[0].plot(df_part3[['sentiment']])
axs[0].set_ylabel("VIX", size=20)
labels=['2019-09-14 to 2020-02-14', '2020-02-15 to 2020-05-31', '2020-06-01 to 2021-07-15']

l1 = axs[0].plot(df_part1[['VIX']])
l2 = axs[0].plot(df_part2[['VIX']])
l3 = axs[0].plot(df_part3[['VIX']])
axs[0].legend([l1, l2, l3], labels=labels, loc='upper right')#, bbox_to_anchor='upper right')#(0.99, 0.85))

axs[1].set_ylabel("Sentiment EWM", size=20)
axs[1].plot(df_part1[['sentiment_ewm']])
axs[1].plot(df_part2[['sentiment_ewm']])
axs[1].plot(df_part3[['sentiment_ewm']])
axs[1].legend([l1, l2, l3], labels=labels, loc='lower right')#, bbox_to_anchor=(0.99, 0.2))

# axs[0].set_ylabel("Sentiment", size=20)
# axs[0].plot(df_part1[['sentiment']])
# axs[0].plot(df_part2[['sentiment']])
# axs[0].plot(df_part3[['sentiment']])


# axs[0].labels()

#plt.subplot(121)
#plt.set_ylabel("VIX", size=20)

# plt.plot(df_part1[['VIX']])

# #plt.subplot(122)
# plt.plot(df_part2[['VIX']])
# # plt.subplot(123)
# plt.plot(df_part3[['VIX']])
# plt.axvspan(x='2020-02-15', color='b', linestyle='--')
starting_date = dt.datetime(2020, 2, 15)
starting_date1 = mdates.date2num(starting_date)
axs[0].axvline(starting_date1, 0, 1, color='b', linestyle='--')
plt.axvline(starting_date1, 0, 1, color='b', linestyle='--')

starting_date2 = dt.datetime(2020, 6, 1)
starting_date3 = mdates.date2num(starting_date2)
axs[0].axvline(starting_date3, 0, 1, color='b', linestyle='--')
plt.axvline(starting_date3, 0, 1, color='b', linestyle='--')



fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/VIXandSentimentSubplots.png")
plt.show()
# import datetime
# start_date = datetime.datetime(2020, 2, 15)
# end_date = datetime.datetime(2020, 6, 1)
# plt.axvspan(start_date, end_date, color='g', alpha=0.5, lw=0)
# fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/VIXandSentimentSubplots2.png")
# plt.show()

In [ ]:
fig, axs = plt.subplots(2,figsize=(20,15))
fig.suptitle('VIX and Sentiment', size=26)
fig.autofmt_xdate()
# axs[0].set_ylabel("sentiment", size=20)
# axs[0].plot(df_part1[['sentiment']])
# axs[0].plot(df_part2[['sentiment']])
# axs[0].plot(df_part3[['sentiment']])
axs[0].set_ylabel("VIX", size=20)
labels=['2019-09-14 to 2020-02-14', '2020-02-15 to 2020-05-31', '2020-06-01 to 2021-07-15']

l1 = axs[0].plot(df_part1[['VIX']])
l2 = axs[0].plot(df_part2[['VIX']])
l3 = axs[0].plot(df_part3[['VIX']])
axs[0].legend([l1, l2, l3], labels=labels, loc='upper right')#, bbox_to_anchor='upper right')#(0.99, 0.85))

axs[1].set_ylabel("Sentiment EWM", size=20)
axs[1].plot(df_part1[['sentiment_ewm']])
axs[1].plot(df_part2[['sentiment_ewm']])
axs[1].plot(df_part3[['sentiment_ewm']])
axs[1].legend([l1, l2, l3], labels=labels, loc='lower right')#, bbox_to_anchor=(0.99, 0.2))

import datetime
start_date = datetime.datetime(2019, 9, 25)
end_date = datetime.datetime(2020, 2, 14)

start_date1 = datetime.datetime(2020, 2, 15)
end_date1 = datetime.datetime(2020, 6, 1)

plt.axvspan(start_date, end_date, color='g', alpha=0.5, lw=0)
axs[0].axvspan(start_date, end_date, color='g', alpha=0.5, lw=0)

plt.axvspan(start_date1, end_date1, color='b', alpha=0.5, lw=0)
axs[0].axvspan(start_date1, end_date1, color='b', alpha=0.5, lw=0)

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/VIXandSentimentSubplots2.png")
plt.show()

In [ ]:
fig, axs = plt.subplots(2,figsize=(25,15))
fig.suptitle(' \n VIX and Sentiment Interplay', size=26)
fig.autofmt_xdate()
# axs[0].set_ylabel("sentiment", size=20)
# axs[0].plot(df_part1[['sentiment']])
# axs[0].plot(df_part2[['sentiment']])
# axs[0].plot(df_part3[['sentiment']])
axs[0].set_ylabel("VIX", size=20)
labels=['2020-06-01 to 2021-07-15']

l1 = axs[0].plot(df_part3[['VIX']])
axs[0].legend([l1, l2, l3], labels=labels, loc='upper right')#, bbox_to_anchor='upper right')#(0.99, 0.85))

axs[1].set_ylabel("Sentiment EWM", size=20)

axs[1].plot(df_part3[['sentiment_ewm']])
axs[1].legend([l1, l2, l3], labels=labels, loc='lower right')#, bbox_to_anchor=(0.99, 0.2))

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/VIXandSentimentSubplots3.png")
plt.show()

In [ ]:
df[['VIX', 'compound', 'neg', 'neu', 'pos', 'sentiment']]

In [ ]:
from plotly import tools
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

# Determine the distribution of charge
vix = df["VIX"].values
sentiment = df["sentiment"].values

# logcharge = np.log(df["VIX"])

trace0 = go.Histogram(
    x=vix,
    histnorm='probability',
    name="VIX Histogram",
    marker = dict(
        color = '#4B8BBE',
    )
)
trace1 = go.Histogram(
    x=sentiment,
    histnorm='probability',
    name="Sentiment Histogram",
    marker = dict(
        color = '#f97306',
    )
)

fig = tools.make_subplots(rows=2, cols=1,
                          subplot_titles=('VI Distribution','Sentiment Distribution'),
                         print_grid=False)



fig.append_trace(trace0, 1, 1)
fig.append_trace(trace1, 2, 1)

fig['layout'].update(showlegend=True, title='VIX and Sentiment Histograms', bargap=0.05)
plot(fig)#, filename='custom-sized-subplot-with-subplot-titles')
#iplot?

In [ ]:
# mu, sigma = 100, 15
# #x = mu + sigma * df['VIX']

# # the histogram of the data
# n, bins, patches = plt.hist(df['VIX'])


# plt.xlabel('Smarts')
# plt.ylabel('Probability')
# plt.title('Histogram of IQ')
# plt.text(100, .025, r'$\mu=100,\ \sigma=15$')
# plt.axis([0, 100, 0, 200])
# plt.grid(True)
# plt.show()

In [ ]:
# fig, axes = plt.subplots(10,2,figsize=(25,75))
# # for i,col in enumerate(df['VIX']):
# sns.regplot(1,'VIX',data=df, ax=axes[i,0]) # regression plot
# sns.kdeplot(df['VIX'],df['sentiment'],shade=True, ax=df[i,1]) # kernel density e

In [ ]:
# # Plot each market variable with VKOSPI

# # Columns to plot 
# plot_columns = ['sentiment','VIX']

# # plot
# fig, axes = plt.subplots(10,2,figsize=(25,75))
# for i,col in enumerate(plot_columns):
#     sns.regplot(col,'VIX',data=df, ax=axes[i,0]) # regression plot
#     sns.kdeplot(df[col],df[['VIX']],shade=True, ax=axes[i,1]) # kernel density estimate plot

In [ ]:
# def cor_reg_kde(x,y,data):
#     # Correlation coefficient
#     #print('correlation coefficient:',corr.loc[x,y])
    
#     # Subplot adjust
#     plt.subplots_adjust(wspace=0.5)
    
#     # Plot (regression and kernel density)
#     fig,(ax1,ax2) = plt.subplots(1,2,figsize=(10,5))
#     sns.regplot(x,y,data=data,ax=ax1)
#     sns.kdeplot(data[[x]],data[y],shade=True,ax=ax2)
    
# cor_reg_kde('pos','VIX',df)
# # cor_reg_kde('Indiv_Put_Netbuying_Quantity','Open_interest',df)
# # cor_reg_kde('Indiv_Put_Netbuying_Quantity','Indiv_Call_Netbuying_Quantity',df)
# # cor_reg_kde('Indiv_Call_Netbuying_Quantity','Open_interest',df)

In [ ]:
# # Checking date for empty strings
# for date, row in df_vix.T.iteritems():
#     if type(df_vix.loc[date, 'articles']).__name__ == 'str':
#         print(date)

In [ ]:
def show_dist(df, col):
    print('Descriptive stats for {}'.format(col))
    print('-'*(len(col)+22))
    print(df.groupby('VIX')[col].describe())
    bins = np.arange(df[col].min(), df[col].max() + 1)
    g = sns.FacetGrid(df, col='VIX', size=5, hue='VIX', palette="PuBuGn_d")
    g = g.map(sns.distplot, col, kde=False, norm_hist=True, bins=bins)
    plt.show()

In [ ]:
#df_vixs=df[['VIX']]
df_vixs=df
df_vixs.plot(x="date", y="sentiment", kind="scatter")
# target_dist = sns.factorplot(x="VIX", data=df_vixs, kind="count", size=10, aspect=1.5, palette="PuBuGn_d")
# plt.show();
# target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/VIX_Histogram.png')

In [ ]:
# # start with simple scatterplot
# sns.scatterplot(x='Day_till_expiration',y='VKOSPI',data=train_org)
# plt.show()

# # too bizarre when VKOSPI is small.
# print('Too bizarre when VKOSPI is small.')
# print('Instead of plotting every data, plot mean of VKOSPI within each Day_till_expiration.')
      
# mean_by_dtexp = train_org.groupby('ExpirationDate').mean()['VKOSPI']
# plt.figure(figsize=(15,5))
# plt.ylabel('Mean of VKOSPI')
# mean_by_dtexp_plot = mean_by_dtexp - 10  # for plotting purpose
# sns.barplot(x=mean_by_dtexp.index,y=mean_by_dtexp_plot.values,bottom=10)
# plt.show()

# # reasonable to suspect that there aren't much data with very high day_till_expiration
# print('It is reasonable to suspect that there aren\'t much data when day_till_expiration is high')
# len_by_dtexp = train_org['Day_till_expiration'].value_counts()
# plt.figure(figsize=(15,5))
# plt.xlabel('Day_till_expiration')
# plt.ylabel('Frequency of Data')
# sns.barplot(x=len_by_dtexp.index,y=len_by_dtexp.values)
# plt.show()

# print('The amount of data seems to be insufficient when Day_till_expiration is high')

In [ ]:
# target_dist = sns.factorplot(x="sentiment", data=df[['sentiment']].round(0), kind="count", size=15, aspect=1.5, palette="PuBuGn_d")
# plt.show();
# target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment_Histogram.png')

In [ ]:
#show_dist(df_vixs, 'VIX')

In [ ]:
# Time series data source: fpp pacakge in R.
import matplotlib.pyplot as plt
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/a10.csv', parse_dates=['date'], index_col='date')

# Draw Plot
def plot_df(df, x, y, title="", xlabel='Date', ylabel='Value', dpi=100):
    plt.figure(figsize=(16,5), dpi=dpi)
    plt.plot(x, y, color='tab:red')
    plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.show()

plot_df(df, x=df.index, y=df.value, title='Monthly anti-diabetic drug sales in Australia from 1992 to 2008.')    

In [ ]:
# Import Data
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/a10.csv', parse_dates=['date'], index_col='date')
df.reset_index(inplace=True)

# Prepare data
df['year'] = [d.year for d in df.date]
df['month'] = [d.strftime('%b') for d in df.date]
years = df['year'].unique()

# Draw Plot
fig, axes = plt.subplots(1, 2, figsize=(20,7), dpi= 80)
sns.boxplot(x='year', y='value', data=df, ax=axes[0])
sns.boxplot(x='month', y='value', data=df.loc[~df.year.isin([1991, 2008]), :])

# Set Title
axes[0].set_title('Year-wise Box Plot\n(The Trend)', fontsize=18); 
axes[1].set_title('Month-wise Box Plot\n(The Seasonality)', fontsize=18)
plt.show()

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,4), dpi=100)
pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/guinearice.csv', parse_dates=['date'], index_col='date').plot(title='Trend Only', legend=False, ax=axes[0])

pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/sunspotarea.csv', parse_dates=['date'], index_col='date').plot(title='Seasonality Only', legend=False, ax=axes[1])

pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/AirPassengers.csv', parse_dates=['date'], index_col='date').plot(title='Trend and Seasonality', legend=False, ax=axes[2])